# F2 · Preprocesamiento de licitaciones de ChileCompra

**Preguntas del proyecto** (ver `F1/F1_Definicion.ipynb`, sección 3):

1. ¿En qué proporción de las licitaciones comparables coincide el monto estimado registrado con el total de las ofertas ganadoras y, cuando difieren, cuál es la magnitud y el sentido de la brecha en cada moneda?
2. ¿Cambian la coincidencia y la brecha según el tamaño del proveedor ganador?

Este notebook prepara los datos para responderlas: la base de ofertas con el tamaño en escala ordinal y la tabla de brecha por licitación.

La brecha describe diferencias entre valores registrados. Para interpretarla como desviación de un presupuesto previo haría falta verificar que el estimado se fijó antes de conocer la adjudicación. La base de ofertas se mantiene a nivel de oferta por ítem. La brecha se guarda en una tabla aparte, con una fila por licitación.

## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías que usa esta fase tienen las versiones declaradas en `requirements.txt`. Es la misma verificación de la sección 8 de `F1_Definicion`, reducida a lo que necesita la Fase 2.

La celda se detiene si no hay entorno virtual o si falta una librería, y avisa si alguna versión no coincide con la declarada.

In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 2, con el nombre que tienen en requirements.txt.
PAQUETES_F2 = ["pandas", "numpy"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas.

    Se detiene si no hay entorno virtual o si falta un paquete. Devuelve una
    tabla con la versión declarada e instalada de cada paquete.
    """
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({"Paquete": paquete, "Declarada": declaradas.get(paquete),
                      "Instalada": instalada, "Coincide": instalada == declaradas.get(paquete)})
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F2)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas: revisa requirements.txt.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")

Intérprete: .venv\Scripts\python.exe | Python 3.15.2 | Windows
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True


[OK] Las 2 versiones coinciden con requirements.txt


## 1. Configurar el proyecto

El archivo de entrada no se versiona. Descárgalo y déjalo en `data/raw/04LicitacionesEne.csv` siguiendo la sección *Obtener los datos* del `README.md` (detalle y verificación SHA-256 en `data/raw/README.md`).

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

actual = Path.cwd()
raiz = next((
    carpeta for carpeta in (actual, *actual.parents)
    if (carpeta / "src/preprocesamiento.py").is_file()
), None)
if raiz is None:
    raise FileNotFoundError("Ubica preprocesamiento.py y pruebas_preprocesamiento.py en src/ del proyecto.")
sys.path.insert(0, str(raiz))
from src.preprocesamiento import (
    FECHAS, MOTIVOS, aplicar_bandera_atipicos, calcular_brecha, calcular_limites_iqr,
    cargar_csv, convertir_tipos, marcar_duplicados, normalizar_codigo,
    ordenar_tamano, perfilar_columnas, tratar_faltantes, validar_procesado,
)
from src.pruebas_preprocesamiento import ejecutar_pruebas_preprocesamiento

ruta = raiz / "data/raw/04LicitacionesEne.csv"
print("Entrada:", ruta.relative_to(raiz))
print("Versión de pandas:", pd.__version__)

Entrada: data\raw\04LicitacionesEne.csv
Versión de pandas: 3.1.0


## 2. Verificar las reglas con datos ficticios

In [3]:
ejecutar_pruebas_preprocesamiento()
print("Pruebas de brecha, duplicados, tamaño, esquema e IQR aprobadas.")

Pruebas de brecha, duplicados, tamaño, esquema e IQR aprobadas.


## 3. Cargar, perfilar y revisar duplicados

- **Filtrado temprano de privacidad (`es_columna_privada`) y separador explícito (`sep=";"`):** Se omiten los identificadores directos del proveedor en la lectura (`72` columnas en crudo → `71` en memoria) porque no aportan al análisis de brechas por tamaño y su exclusión temprana protege los datos personales antes de perfilar o exportar.
- **Detección de duplicados potenciales (`EsDuplicadoPotencial`):** Se marcan las coincidencias exactas sobre esas `71` columnas **antes de convertir tipos o recortar variables** para distinguir ofertas legítimamente repetidas de duplicados administrativos. Se conservan todas las filas (`45.741`) y posteriormente se evalúa la sensibilidad excluyendo las licitaciones con ganadoras potencialmente duplicadas (`11` filas adicionales en ganadoras y `11` en perdedoras) en lugar de eliminar registros sin evidencia causal.

In [4]:
datos = cargar_csv(ruta, verificar_hash=True)
filas_originales = len(datos)
print(f"Entrada: {filas_originales:,} filas y {datos.shape[1]} columnas")
perfil = perfilar_columnas(datos)
display(perfil)
nulos_entrada = datos.isna().sum()
datos, resumen_duplicados = marcar_duplicados(datos)
print("Filas adicionales idénticas por resultado:")
display(resumen_duplicados)

Entrada: 45,741 filas y 71 columnas


,Tipo,Nulos,% nulos,Valores distintos
NroLicitacion,string,0,0.0,2292
NombreLicitacion,str,0,0.0,2282
TipoLicitacion,str,0,0.0,9
Descripcion,str,0,0.0,2222
MonedaLicitacion,str,0,0.0,4
...,...,...,...,...
CantidadOferta,int64,0,0.0,358
MonedaOferta,str,0,0.0,4
MontoNetoOferta,str,0,0.0,16765
MontoTotalOferta,str,0,0.0,21376


Filas adicionales idénticas por resultado:


,FilasEnGrupoIdentico,FilasAdicionales
ResultadoOferta,,
Ganadora,22,11
Perdedora,22,11


## 4. Convertir y decidir qué hacer con los faltantes (*casting* y normalización)

Cada transformación aplicada en esta etapa responde a un requerimiento técnico concreto de los datos de entrada y a las dos preguntas de investigación:

1. **Casting de fechas (`FechaPublicacion`, `FechaCierre`, `FechaAdjudicacion` → `datetime64[ns]`):** En el CSV original llegan como texto (`%d-%m-%Y %H:%M`). El *casting* con formato explícito evita ambigüedades día/mes, habilita la validación cronológica de rangos y distingue fallos reales de conversión (`0` fallos) respecto de las ausencias legítimas (`91` filas sin `FechaAdjudicacion`, `0,20 %`, correspondientes a licitaciones aún no adjudicadas).
2. **Casting numérico de montos (`MontoNetoOferta`, `MontoTotalOferta`, `MontoEstimadoLicitacion` → `Float64`):** Las ofertas traen coma decimal como separador en texto plano (`str`). Se reemplaza `,` por `.` y se convierte a flotante *nullable* (`Float64`), paso indispensable para sumar ofertas ganadoras, calcular brechas exactas y aplicar el logaritmo en IQR sin redondear ni ocultar posibles valores nulos (`0` fallos de conversión en las `45.741` filas).
3. **Normalización de cadenas en blanco (`r"^\s*$"` → `pd.NA`) y criterio sobre faltantes:** Unifica espacios vacíos como ausencias explícitas antes de medir completitud. En las `14` variables analíticas (`MOTIVOS`) **se conservan los faltantes sin imputación artificial**, porque inventar fechas, montos o resultados distorsionaría la brecha y la tasa de coincidencia; las columnas fuera del análisis (incluidas las `7` variables administrativas con `> 90 %` de faltantes) se documentan aparte y se excluyen al exportar.
4. **Normalización de códigos (`normalizar_codigo`) y escala ordinal (`TamanoProveedor` → `TamanoProveedorOrdinal`):** Limpia espacios exteriores y unifica mayúsculas en códigos clave (`NroLicitacion`, monedas, `ResultadoOferta`, `MontoEstimadoVisible`) para evitar separaciones espurias al agrupar. Para el tamaño de empresa, se impone la jerarquía ordenada `Micro < Pequeña < Mediana < Grande` necesaria para responder la Pregunta 2, dejando `NoClasificado` (`5.274` ofertas) fuera de la escala como `pd.NA` porque representa ausencia de clasificación y no un rango empresarial.

In [5]:
datos, conversiones = convertir_tipos(datos)
display(conversiones)
datos, faltantes = tratar_faltantes(datos, nulos_entrada.reindex(datos.columns))
faltantes = faltantes.drop(index="EsDuplicadoPotencial")
display(faltantes.loc[faltantes.index.intersection(MOTIVOS)])
display(faltantes.loc[faltantes["% faltantes"].gt(90)].sort_values("% faltantes", ascending=False))
datos = ordenar_tamano(datos)
display(datos["TamanoProveedorOrdinal"].value_counts(dropna=False).sort_index())

,Nulos en entrada,Fallos de conversión
Columna,,
FechaPublicacion,0,0
FechaCierre,0,0
FechaAdjudicacion,91,0
MontoNetoOferta,0,0
MontoTotalOferta,0,0
MontoEstimadoLicitacion,0,0


,Nulos al leer,Nulos después de convertir y normalizar,% faltantes,Regla,Motivo
NroLicitacion,0.0,0,0.000000,Conservar faltantes,"Identifica la licitación. Sin código, la fila ..."
MonedaLicitacion,0.0,0,0.000000,Conservar faltantes,"Unidad del estimado. Si falta, no puede verifi..."
MontoEstimadoLicitacion,0.0,0,0.000000,Conservar faltantes,Referencia de comparación. Sin monto válido se...
MontoEstimadoVisible,0.0,0,0.000000,Conservar faltantes,Distingue visibilidad pública del monto; vacío...
FechaPublicacion,0.0,0,0.000000,Conservar faltantes,Permite describir cuándo se publicó. Se descon...
FechaCierre,0.0,0,0.000000,Conservar faltantes,Permite describir el plazo de recepción cuando...
FechaAdjudicacion,91.0,91,0.198946,Conservar faltantes,Permite estudiar fechas de adjudicación observ...
CodigoProductoONU,0.0,0,0.000000,Conservar faltantes,Permite segmentar por producto cuando hay código.
TamanoProveedor,0.0,0,0.000000,Conservar faltantes,Describe tamaño. Una ausencia queda sin rango ...
CantidadOferta,0.0,0,0.000000,Conservar faltantes,Permite contrastar el cálculo del monto total ...


,Nulos al leer,Nulos después de convertir y normalizar,% faltantes,Regla,Motivo
LicitacionBaseTipo,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.
UnidadTiempoRenovacion,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.
ContratoRenovable,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.
FechaEntregaEnSoporteFisico,45457.0,45457,99.379113,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
FechaEstimadaFirmaContrato,45270.0,45270,98.970289,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
RazonPublicidadOfertasTecnicas,44018.0,44018,96.233139,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
FechaEstimadaEvaluacionOfertas,43489.0,43489,95.076627,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...


TamanoProveedorOrdinal
Micro       6539
Pequeña    13510
Mediana     8570
Grande     11848
NaN         5274
Name: count, dtype: int64

### 4.1 Revisar fechas, monedas y categorías después de convertir

Se comprueba el resultado de las conversiones antes de calcular la brecha:

- **Fechas.** El rango de cada fecha principal debe ser coherente con el corte de enero de 2025: todas las licitaciones cierran en enero, la publicación puede ser anterior (desde noviembre de 2024) y la adjudicación, posterior. Las 91 ofertas sin fecha de adjudicación son las ausencias originales que la sección 4 ya informó.
- **Monedas.** La frecuencia por moneda justifica separar los montos: CLP concentra casi todas las ofertas, y CLF, USD y UTM no se mezclan con él.
- **Categorías.** Se revisan la modalidad (`TipoLicitacion`), el resultado de la oferta y el tamaño de proveedor. `NoClasificado` no es un tamaño: queda fuera de la escala ordinal, como faltante.

In [6]:
# Fechas: rango observado después de convertir.
display(pd.DataFrame({
    "Mínima": datos[list(FECHAS)].min(),
    "Máxima": datos[list(FECHAS)].max(),
    "Sin fecha": datos[list(FECHAS)].isna().sum(),
}))

# Monedas: ofertas por moneda de la oferta y de la licitación.
display(pd.concat({
    "Ofertas (MonedaOferta)": datos["MonedaOferta"].value_counts(),
    "Ofertas (MonedaLicitacion)": datos["MonedaLicitacion"].value_counts(),
}, axis=1).fillna(0).astype(int))

# Categorías: modalidad, resultado y tamaño (original frente a escala ordinal).
display(datos["TipoLicitacion"].value_counts().to_frame("Ofertas"))
display(datos["ResultadoOferta"].value_counts().to_frame("Ofertas"))
escala = list(datos["TamanoProveedorOrdinal"].cat.categories)
tamanos = pd.crosstab(datos["TamanoProveedor"], datos["TamanoProveedorOrdinal"].astype("string").fillna("Sin rango (NaN)"))
display(tamanos.reindex(index=[*escala, "NoClasificado"], columns=[*escala, "Sin rango (NaN)"], fill_value=0))


,Mínima,Máxima,Sin fecha
FechaPublicacion,2024-11-13 13:41:00,2025-01-25 19:16:00,0
FechaCierre,2025-01-02 15:00:00,2025-01-31 20:00:00,0
FechaAdjudicacion,2025-01-03 09:45:00,2025-04-22 17:22:00,91


,Ofertas (MonedaOferta),Ofertas (MonedaLicitacion)
CLP,45604,45604
CLF,105,105
USD,23,23
UTM,9,9


,Ofertas
TipoLicitacion,
Licitación Pública Entre 100 y 1000 UTM (LE),32569
Licitación Pública Menor a 100 UTM (L1),7594
Licitación Pública entre a 2000 y 5000 UTM (LQ),2026
Licitación Pública Mayor a 5000 (LR),1836
Licitación Pública Mayor 1000 UTM (LP),1664
Licitación Privada entre 100 y 1000 UTM.,31
Licitación Privada Menor a 100 UTM.,10
Licitación Privada Mayor a 1000 UTM,6
Licitación Pública Servicios personales especializados (LS),5


,Ofertas
ResultadoOferta,
Perdedora,25853
Ganadora,19888


TamanoProveedorOrdinal,Micro,Pequeña,Mediana,Grande,Sin rango (NaN)
TamanoProveedor,,,,,
Micro,6539,0,0,0,0
Pequeña,0,13510,0,0,0
Mediana,0,0,8570,0,0
Grande,0,0,0,11848,0
NoClasificado,0,0,0,0,5274


## 5. Calcular la brecha por licitación

**Justificación de agregación y no conversión monetaria:** Se suman los montos totales de **todas las ofertas ganadoras** por licitación y se exige coherencia monetaria estricta (`MonedaOferta == MonedaLicitacion`, estimado positivo y consistente, y ausencia de resultados faltantes). **No se convierten ni reexpresan monedas entre sí (`CLP`, `CLF`, `USD`, `UTM`)**, porque la base no registra el tipo de cambio histórico del día de apertura y aplicar un factor promedio introduciría sesgos artificiales en la brecha.

**Brecha = suma de montos ganadores − monto estimado registrado.** Se reporta la cobertura por licitación (`2.257` comparables de `2.292`, `98,5 %`) y se resume por moneda en sus unidades originales. Para identificar coincidencias se usa una tolerancia absoluta de `1e-8` por redondeo en punto flotante, contrastando la sensibilidad frente a ganadoras duplicadas potenciales.

In [7]:
licitaciones = calcular_brecha(datos)
comparables = licitaciones.loc[licitaciones["EstadoBrecha"].eq("Calculada")]
sin_duplicadas = comparables.loc[~comparables["GanadoraDuplicada"]]

print(f"Licitaciones con código: {len(licitaciones):,}")
print(f"Filas sin código: {normalizar_codigo(datos['NroLicitacion']).isna().sum():,}")
display(licitaciones["EstadoBrecha"].value_counts().to_frame("Licitaciones"))
print(f"Brechas calculadas: {len(comparables):,} de {len(licitaciones):,}")
if len(licitaciones):
    print(f"Cobertura: {len(comparables) / len(licitaciones):.1%}")
for nombre, muestra in [("Todas las comparables", comparables), ("Sin ganadoras duplicadas potenciales", sin_duplicadas)]:
    print(nombre)
    display(muestra.groupby("MonedaLicitacion").agg(
        Licitaciones=("BrechaMonto", "size"),
        Coinciden=("CoincideConEstimado", "sum"),
        ProporcionCoincide=("CoincideConEstimado", "mean"),
        MedianaBrecha=("BrechaMonto", "median"),
    ))

Licitaciones con código: 2,292
Filas sin código: 0


,Licitaciones
EstadoBrecha,
Calculada,2257
Sin ofertas ganadoras,26
Estimado inválido o inconsistente,8
Ganadora con monto inválido,1


Brechas calculadas: 2,257 de 2,292
Cobertura: 98.5%
Todas las comparables


,Licitaciones,Coinciden,ProporcionCoincide,MedianaBrecha
MonedaLicitacion,,,,
CLF,20,0,0.0,-60.36
CLP,2230,1252,0.561435,0.0
USD,3,2,0.666667,0.0
UTM,4,0,0.0,-31.64


Sin ganadoras duplicadas potenciales


,Licitaciones,Coinciden,ProporcionCoincide,MedianaBrecha
MonedaLicitacion,,,,
CLF,20,0,0.0,-60.36
CLP,2226,1252,0.562444,0.0
USD,3,2,0.666667,0.0
UTM,4,0,0.0,-31.64


## 6. Revisar si el estimado sirve como referencia previa

El [diccionario de licitaciones de la API de Mercado Público](https://www.chilecompra.cl/wp-content/uploads/2026/03/Documentacion-API-Mercado-Publico-Licitaciones.pdf) describe el monto estimado de la institución y un indicador que determina si se publica en la ficha. Es documentación de la API. No establece cuándo se registró el valor en este extracto CSV ni confirma su independencia de lo adjudicado.

Se contrasta la coincidencia por moneda y visibilidad. Los campos `BaseEstimacionMontoLicitacion` y `JustificacionMontoEstimado` se examinan mediante conteos de presencia y menciones a adjudicación o presupuesto. Estas menciones orientan la revisión y requieren interpretación del equipo. Los textos se quedan en el CSV local y no se muestran ni se exportan.

In [8]:
if "MontoEstimadoVisible" in datos:
    visible = normalizar_codigo(datos["MontoEstimadoVisible"]).str.replace("Í", "I")
    print("Filas de oferta con monto indicado como no visible:", visible.eq("NO").sum())
    print("Proporción del total de filas:", round(visible.eq("NO").sum() / len(datos), 4) if len(datos) else 0)
    display(comparables.groupby(["MonedaLicitacion", "VisibilidadEstimado"]).agg(
        Licitaciones=("BrechaMonto", "size"),
        Coinciden=("CoincideConEstimado", "sum"),
        ProporcionCoincide=("CoincideConEstimado", "mean"),
    ))

campos_contexto = [c for c in licitaciones if c.endswith((
    "_Informada", "_MencionaAdjudicacion", "_MencionaPresupuesto",
))]
if campos_contexto:
    display(licitaciones[campos_contexto].sum().rename("Licitaciones").to_frame())
    display(comparables.groupby("CoincideConEstimado")[campos_contexto].sum())

Filas de oferta con monto indicado como no visible: 24895
Proporción del total de filas: 0.5443


Licitaciones  Coinciden  \
MonedaLicitacion VisibilidadEstimado                            
CLF              NO                             10          0   
                 SI                             10          0   
CLP              NO                           1189        635   
                 SI                           1041        617   
USD              NO                              1          1   
                 SI                              2          1   
UTM              NO                              4          0   

                                      ProporcionCoincide  
MonedaLicitacion VisibilidadEstimado                      
CLF              NO                                  0.0  
                 SI                                  0.0  
CLP              NO                             0.534062  
                 SI                             0.592699  
USD              NO                                  1.0  
                 SI                                  0.5  
UTM              NO                                  0.0

,Licitaciones
BaseEstimacionMontoLicitacion_Informada,2292
BaseEstimacionMontoLicitacion_MencionaAdjudicacion,0
BaseEstimacionMontoLicitacion_MencionaPresupuesto,1603
JustificacionMontoEstimado_Informada,800
JustificacionMontoEstimado_MencionaAdjudicacion,3
JustificacionMontoEstimado_MencionaPresupuesto,264


,BaseEstimacionMontoLicitacion_Informada,BaseEstimacionMontoLicitacion_MencionaAdjudicacion,BaseEstimacionMontoLicitacion_MencionaPresupuesto,JustificacionMontoEstimado_Informada,JustificacionMontoEstimado_MencionaAdjudicacion,JustificacionMontoEstimado_MencionaPresupuesto
CoincideConEstimado,,,,,,
False,1003,0,666,313,2,105
True,1254,0,907,475,1,155


## 7. Escalamiento de variables y detección de valores extremos con IQR sobre logaritmos

### Por qué NO se aplica escalamiento general (*Min-Max* ni *Z-score*)
- **Naturaleza descriptiva del estudio:** El proyecto no entrena modelos predictivos sensibles a la magnitud de las variables (como KNN, SVM, regresión regularizada o redes neuronales), por lo que reescalar a `[0, 1]` o estandarizar a media `0` y desviación `1` no aporta beneficio algorítmico.
- **Preservación de unidades monetarias reales:** Las preguntas de investigación evalúan la **brecha absoluta en moneda original** (`BrechaMonto`) y la proporción de coincidencia exacta (`Brecha == 0`) dentro de cada divisa (`CLP`, `CLF`, `USD`, `UTM`). Escalar los montos destruiría su interpretabilidad económica y ocultaría la diferencia real entre desviaciones menores y millones de pesos.

### Por qué SÍ se usa transformación logarítmica acotada en el detector IQR
- **Asimetría extrema de montos públicos:** En `CLP` coexisten licitaciones menores (`L1`, `< 100 UTM`) y grandes contratos (`LR`, `> 5.000 UTM`). Aplicar IQR sobre la escala lineal genera límites inferiores negativos sin sentido económico y marca un exceso de ofertas altas legítimas.
- **Transformación no destructiva (`np.log` solo para límites):** Se aplica $\ln(\text{MontoTotalOferta})$ únicamente sobre ofertas en `CLP` con monto positivo (`45.603` filas) para estabilizar proporciones y calcular los límites $Q_1 - 1,5 \times \text{IQR}$ y $Q_3 + 1,5 \times \text{IQR}$. Los umbrales se reportan reexpresados en pesos reales ($\exp(\cdot)$) y el resultado se exporta como bandera booleana (`EsExtremoLogIQR_CLP`), **conservando intactos todos los montos originales y las `45.741` filas**.

In [9]:
limites = calcular_limites_iqr(datos)
datos = aplicar_bandera_atipicos(datos, limites)
if limites is not None:
    inferior = np.exp(limites["inferior_log"])
    superior = np.exp(limites["superior_log"])
    evaluadas = datos["EsExtremoLogIQR_CLP"].notna()
    montos = datos.loc[evaluadas, "MontoTotalOferta"]
    display(pd.DataFrame([{
        **limites, "Límite inferior CLP": inferior, "Límite superior CLP": superior,
        "Extremos inferiores": int(np.log(montos).lt(limites["inferior_log"]).sum()),
        "Extremos superiores": int(np.log(montos).gt(limites["superior_log"]).sum()),
    }]))
display(datos["EsExtremoLogIQR_CLP"].value_counts(dropna=False).to_frame("Ofertas"))

,ofertas_validas,q1_log,q3_log,inferior_log,superior_log,Límite inferior CLP,Límite superior CLP,Extremos inferiores,Extremos superiores
0,45603,10.683179,14.404324,5.101462,19.986041,164.261932,4.784398e+08,810,77


,Ofertas
EsExtremoLogIQR_CLP,
False,44716
True,887
<NA>,138


## 8. Síntesis de transformaciones, validación y exportación

| Etapa | Variable(s) | Tipo de transformación | Regla aplicada | Justificación e impacto en objetivos |
| :--- | :--- | :--- | :--- | :--- | 
| Lectura (Sec. 3) | Campos privados de proveedor | Filtrado / Privacidad | Omite identificadores directos con `es_columna_privada()` | Protege datos personales desde el inicio (`72` → `71` columnas) sin afectar la agregación |
| Duplicados (Sec. 3) | Fila completa sin IDs | Marcado preventivo | Señala duplicados exactos en `EsDuplicadoPotencial` | Permite análisis de sensibilidad sobre `2.257` vs. `2.253` licitaciones sin borrar datos |
| Fechas (Sec. 4) | `FechaPublicacion`, `FechaCierre`, `FechaAdjudicacion` | **Casting** | Texto `%d-%m-%Y %H:%M` → `datetime64[ns]` (`0` fallos) | Garantiza orden cronológico verificable preservando `91` nulos originales |
| Montos (Sec. 4) | `MontoNetoOferta`, `MontoTotalOferta`, `MontoEstimadoLicitacion` | **Casting** | Coma decimal → punto y conversión a `Float64` (`0` fallos) | Habilita sumas y brechas exactas manteniendo nulos tipados sin redondear |
| Faltantes y códigos (Sec. 4) | Cadenas en blanco y códigos clave | **Normalización** | `^\s*$` → `pd.NA`; `strip().upper()`; conservar `14` variables `MOTIVOS` | Evita ausencias ocultas o categorías duplicadas por formato sin imputar cifras sintéticas |
| Tamaño empresa (Sec. 4) | `TamanoProveedor` → `TamanoProveedorOrdinal` | **Normalización ordinal** | `Micro < Pequeña < Mediana < Grande`; `NoClasificado` → `pd.NA` | Hace comparable el tamaño empresarial (Pregunta 2) sin distorsionar con `5.274` sin clasificar |
| Montos y brecha (Sec. 5 y 7) | Montos en `CLP`, `CLF`, `USD`, `UTM` | **No escalamiento general** + log temporal en IQR | Conserva unidad monetaria original; usa `np.log` solo para límites `EsExtremoLogIQR_CLP` | Mantiene interpretabilidad económica de `BrechaMonto` (Pregunta 1) mientras controla asimetría al marcar `887` extremos |

A continuación `validar_procesado()` comprueba invariantes (cardinalidad, columnas permitidas, escala ordinal, cobertura de brecha y bandera de extremos) y exporta `04LicitacionesEne_procesado.csv` y `04LicitacionesEne_brecha_por_licitacion.csv`.

In [10]:
columnas_ofertas = [c for c in MOTIVOS if c in datos]
columnas_ofertas += ["EsDuplicadoPotencial", "TamanoProveedorOrdinal", "EsExtremoLogIQR_CLP"]
ofertas = datos[columnas_ofertas]
for comprobacion in validar_procesado(ofertas, licitaciones, filas_originales, limites):
    print(f"[OK] {comprobacion}")

salida = raiz / "data/processed"
salida.mkdir(parents=True, exist_ok=True)
for tabla, nombre in [
    (ofertas, "04LicitacionesEne_procesado.csv"),
    (licitaciones, "04LicitacionesEne_brecha_por_licitacion.csv"),
]:
    destino = salida / nombre
    temporal = destino.with_suffix(".csv.tmp")
    tabla.to_csv(temporal, sep=";", index=False, encoding="utf-8-sig", lineterminator="\n")
    temporal.replace(destino)
    print(f"{destino.relative_to(raiz)}: {len(tabla):,} filas × {tabla.shape[1]} columnas")

[OK] Ofertas: se conservan las 45,741 filas de entrada
[OK] Ofertas: las 17 columnas pertenecen al conjunto autorizado
[OK] Licitaciones: 2,292 códigos únicos, los mismos de las ofertas
[OK] Tamaño: escala Micro < Pequeña < Mediana < Grande; NoClasificado sin rango
[OK] Brecha: existe solo en las licitaciones con estado Calculada
[OK] Brecha: montos ganadores y estimados positivos, en la misma moneda
[OK] Extremos: la bandera existe solo en ofertas CLP con monto positivo
data\processed\04LicitacionesEne_procesado.csv: 45,741 filas × 17 columnas
data\processed\04LicitacionesEne_brecha_por_licitacion.csv: 2,292 filas × 24 columnas
